# Consolidated Tables & LaTeX Analysis Pipeline

Unified pipeline to discover, filter, and export LaTeX tables from `Plots/**/Tables/*.csv`:
- Automatically extracts `ModelsCompared` (e.g. `Model_V10_100_Rings_vs_Athena_Cut_Loose`), `ComparisonRun`, `ET`, and `ETA`.
- Each table group follows the same workflow: **Load -> Filter -> Format & Export LaTeX**.

In [16]:
%load_ext autoreload
%autoreload 2

import re
import sys
from pathlib import Path
from typing import Optional

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.core.Validation.DistanceMetricsETL import DistanceMetricsETL
from src.core.Validation.LatexTableFormatter import LatexTableFormatter

print(f"Project root: {PROJECT_ROOT}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root: /eos/home-i04/p/pmourafr/egamma_master


## 1. Folder Configuration & Shared Helpers

In [17]:
PLOTS_ROOT: Path = PROJECT_ROOT / "Plots"
INPUT_FOLDERS: list[Path] = [
    # PLOTS_ROOT,
    PROJECT_ROOT / "Plots/Model_V10_100_Rings_vs_Model_V10_50_Rings",
    # PROJECT_ROOT / "Plots/Model_V10_100_Rings_vs_Model_V10_50_Rings",
]

OUTPUT_DIR: Path = PLOTS_ROOT / "ConsolidatedAnalysis"
TABLES_DIR: Path = OUTPUT_DIR / "Tables"
LATEX_DIR: Path = OUTPUT_DIR / "LaTeX"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
LATEX_DIR.mkdir(parents=True, exist_ok=True)

def _parse_models_and_run(csv_path: Path, plots_root: Path = PLOTS_ROOT) -> tuple[str, str]:
    """Extracts (ModelsCompared, ComparisonRun) from Plots/<ModelsCompared>/<ComparisonRun>/Tables/<file>.csv."""
    try:
        rel = csv_path.relative_to(plots_root).parts
    except ValueError:
        rel = csv_path.parts
    if len(rel) >= 4 and rel[-2] == "Tables":
        return rel[-4], rel[-3]
    if len(rel) >= 3 and rel[-2] == "Tables":
        return rel[-3], rel[-3]
    return csv_path.parent.name, csv_path.parent.name

def _parse_et_eta(text: str) -> tuple[Optional[int], Optional[int]]:
    m = re.search(r"et_?(\d+).*?eta_?(\d+)", text)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

def _discover_csvs_in_folders(folders: list[Path], file_pattern: str) -> list[Path]:
    """Discovers matching CSVs across a list of base, model-pair, run, or Tables folders."""
    seen: set[Path] = set()
    discovered: list[Path] = []
    for folder in folders:
        folder = Path(folder)
        if not folder.exists():
            print(f"Skipping missing folder: {folder}")
            continue
        matches = list(folder.glob(file_pattern)) if folder.name == "Tables" else list(folder.glob(f"**/Tables/{file_pattern}"))
        for p in sorted(matches):
            resolved = p.resolve()
            if not p.name.startswith(".") and not any(part.startswith(".sys.v#") for part in p.parts) and "ConsolidatedAnalysis" not in resolved.parts and resolved not in seen:
                seen.add(resolved)
                discovered.append(p)
    return sorted(discovered)

def load_table_group(file_pattern: str, folders: list[Path] = INPUT_FOLDERS, plots_root: Path = PLOTS_ROOT) -> pd.DataFrame:
    """Discovers and loads all CSVs matching `file_pattern` across `folders`, adding ModelsCompared."""
    csv_files = _discover_csvs_in_folders(folders, file_pattern)
    print(f"[{file_pattern}] Discovered {len(csv_files)} files across {len(folders)} folder(s)")
    frames = []
    for p in csv_files:
        part = pd.read_csv(p)
        models_compared, comp_run = _parse_models_and_run(p, plots_root)
        et, eta = _parse_et_eta(p.name)
        part.insert(0, "ModelsCompared", models_compared)
        part.insert(1, "ComparisonRun", comp_run)
        part.insert(2, "ET", et)
        part.insert(3, "ETA", eta)
        part["SourceFile"] = p.name
        frames.append(part)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def apply_common_filters(
    df: pd.DataFrame,
    models_filter: Optional[list[str]] = None,
    run_contains: Optional[str] = None,
    et_filter: Optional[list[int]] = None,
    eta_filter: Optional[list[int]] = None,
) -> pd.DataFrame:
    """Applies standard ModelsCompared, ComparisonRun, ET, and ETA filters."""
    out = df.copy()
    if models_filter:
        out = out[out["ModelsCompared"].isin(models_filter)]
    if run_contains:
        out = out[out["ComparisonRun"].str.contains(run_contains, na=False)]
    if et_filter:
        out = out[out["ET"].isin(et_filter)]
    if eta_filter:
        out = out[out["ETA"].isin(eta_filter)]
    return out.reset_index(drop=True)

def save_group_outputs(df_raw: pd.DataFrame, df_latex: pd.DataFrame, group_name: str, caption: str, label: str) -> str:
    """Saves consolidated CSV and formatted LaTeX table under Plots/ConsolidatedAnalysis/."""
    csv_path = TABLES_DIR / f"consolidated_{group_name}.csv"
    tex_path = LATEX_DIR / f"consolidated_{group_name}_latex_table.tex"
    df_raw.to_csv(csv_path, index=False)
    latex_str = LatexTableFormatter.to_latex(df_latex, caption=caption, label=label)
    tex_path.write_text(latex_str, encoding="utf-8")
    print(f"Saved CSV:   {csv_path}")
    print(f"Saved LaTeX: {tex_path}")
    return latex_str

available_runs = sorted({_parse_models_and_run(p, PLOTS_ROOT)[0] for p in _discover_csvs_in_folders(INPUT_FOLDERS, "*.csv")})
print("Available model comparison folders:", available_runs)

Available model comparison folders: ['Model_V10_100_Rings_vs_Model_V10_50_Rings']


## 2. Group 1 — Variable JS Distance (`*_variable_js_divergence_*.csv`)

In [18]:
# 2a. Load
df_js = load_table_group("*_variable_js_divergence_*.csv")
display(df_js.head(10))

[*_variable_js_divergence_*.csv] Discovered 4 files across 1 folder(s)


,ModelsCompared,ComparisonRun,ET,ETA,Feature,Reference,Strategy,Comparison,Ref Count,Selected Count,JS Distance (sqrt(D_JS)),JS Distance (sqrt(D_JS)) Uncertainty,SourceFile
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_et,True Background,Model V10 (100 Rings) [Best SP],Model V10 (100 Rings) [Best SP] vs True Backgr...,10384,1412,0.082280,0.011563,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_et,True Background,Model V10 (50 Rings) [Best SP],Model V10 (50 Rings) [Best SP] vs True Background,10384,1579,0.078461,0.010299,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...
2,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_et,True Background,Athena Cut (Loose),Athena Cut (Loose) vs True Background,10384,4309,0.049857,0.006720,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...
3,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_eta,True Background,Model V10 (100 Rings) [Best SP],Model V10 (100 Rings) [Best SP] vs True Backgr...,10384,1412,0.060170,0.011886,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...
4,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_eta,True Background,Model V10 (50 Rings) [Best SP],Model V10 (50 Rings) [Best SP] vs True Background,10384,1579,0.061900,0.009197,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...
5,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_eta,True Background,Athena Cut (Loose),Athena Cut (Loose) vs True Background,10384,4309,0.104842,0.007146,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...
6,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_phi,True Background,Model V10 (100 Rings) [Best SP],Model V10 (100 Rings) [Best SP] vs True Backgr...,10384,1412,0.063349,0.010701,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...
7,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_phi,True Background,Model V10 (50 Rings) [Best SP],Model V10 (50 Rings) [Best SP] vs True Background,10384,1579,0.057108,0.009405,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...
8,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_phi,True Background,Athena Cut (Loose),Athena Cut (Loose) vs True Background,10384,4309,0.033774,0.006735,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...
9,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_e237,True Background,Model V10 (100 Rings) [Best SP],Model V10 (100 Rings) [Best SP] vs True Backgr...,10384,1412,0.168277,0.011280,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...


In [19]:
# 2b. Filter
JS_MODELS_FILTER: Optional[list[str]] = None     # e.g. ["Model_V10_100_Rings_vs_Athena_Cut_Loose"]
JS_RUN_CONTAINS: Optional[str] = None            # e.g. "best_sp_vs_athena_cut"
JS_FEATURES_FILTER: Optional[list[str]] = ["Eratio", "trig_L2_calo_et", "trig_L2_calo_weta2"]
JS_REFERENCE_FILTER: Optional[str] = None        # e.g. "True Signal" or "True Background"

df_js_filt = apply_common_filters(df_js, models_filter=JS_MODELS_FILTER, run_contains=JS_RUN_CONTAINS)
if not df_js_filt.empty:
    if JS_FEATURES_FILTER and "Feature" in df_js_filt.columns:
        df_js_filt = df_js_filt[df_js_filt["Feature"].isin(JS_FEATURES_FILTER)]
    if JS_REFERENCE_FILTER and "Reference" in df_js_filt.columns:
        df_js_filt = df_js_filt[df_js_filt["Reference"] == JS_REFERENCE_FILTER]
    df_js_filt = DistanceMetricsETL.evaluate_distribution_equivalence(df_js_filt)
display(df_js_filt)

,ModelsCompared,ComparisonRun,ET,ETA,Feature,Reference,Strategy,Comparison,Ref Count,Selected Count,JS Distance (sqrt(D_JS)),JS Distance (sqrt(D_JS)) Uncertainty,SourceFile,JS_Equivalent
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_et,True Background,Model V10 (100 Rings) [Best SP],Model V10 (100 Rings) [Best SP] vs True Backgr...,10384,1412,0.082280,0.011563,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...,False
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_et,True Background,Model V10 (50 Rings) [Best SP],Model V10 (50 Rings) [Best SP] vs True Background,10384,1579,0.078461,0.010299,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...,False
2,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_et,True Background,Athena Cut (Loose),Athena Cut (Loose) vs True Background,10384,4309,0.049857,0.006720,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...,False
18,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_weta2,True Background,Model V10 (100 Rings) [Best SP],Model V10 (100 Rings) [Best SP] vs True Backgr...,10384,1412,0.360072,0.011633,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...,False
19,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_weta2,True Background,Model V10 (50 Rings) [Best SP],Model V10 (50 Rings) [Best SP] vs True Background,10384,1579,0.355466,0.010696,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...,False
20,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_weta2,True Background,Athena Cut (Loose),Athena Cut (Loose) vs True Background,10384,4309,0.251021,0.006784,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...,False
33,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Eratio,True Background,Model V10 (100 Rings) [Best SP],Model V10 (100 Rings) [Best SP] vs True Backgr...,10384,1412,0.453091,0.010820,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...,False
34,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Eratio,True Background,Model V10 (50 Rings) [Best SP],Model V10 (50 Rings) [Best SP] vs True Background,10384,1579,0.455988,0.010484,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...,False
35,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Eratio,True Background,Athena Cut (Loose),Athena Cut (Loose) vs True Background,10384,4309,0.084317,0.006769,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...,False
36,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,trig_L2_calo_et,True Signal,Model V10 (100 Rings) [Best SP],Model V10 (100 Rings) [Best SP] vs True Signal,3994,3205,0.058938,0.008456,et_1_eta_1_best_sp_vs_best_sp_variable_js_dive...,False


In [20]:
# 2c. Format & Export LaTeX
if not df_js_filt.empty:
    js_col = "JS Distance (sqrt(D_JS))" if "JS Distance (sqrt(D_JS))" in df_js_filt.columns else "JS Divergence"
    js_unc_col = "JS Distance (sqrt(D_JS)) Uncertainty" if "JS Distance (sqrt(D_JS)) Uncertainty" in df_js_filt.columns else "JS Uncertainty"
    js_keep = ["ModelsCompared", "ET", "ETA", "Feature", "Comparison", js_col, js_unc_col]
    js_sub = df_js_filt[[c for c in js_keep if c in df_js_filt.columns]]
    js_fmt = LatexTableFormatter.merge_uncertainty_pairs(
        js_sub,
        pairs={"JS Distance ($\\sqrt{D_{\\text{JS}}}$)": (js_col, js_unc_col)},
        sig_figs=2,
        drop_original=True,
    )
    display(js_fmt)
    js_tex = save_group_outputs(
        df_js_filt, js_fmt, "variable_js_distance",
        caption="Variable distribution JS distance ($\\sqrt{D_{\\text{JS}}}$) across comparisons.",
        label="tab:variable_js_distance",
    )
    print(js_tex)

,ModelsCompared,ET,ETA,Feature,Comparison,JS Distance ($\sqrt{D_{\text{JS}}}$)
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,trig_L2_calo_et,Model V10 (100 Rings) [Best SP] vs True Backgr...,$0.082 \pm 0.012$
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,trig_L2_calo_et,Model V10 (50 Rings) [Best SP] vs True Background,$0.078 \pm 0.010$
2,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,trig_L2_calo_et,Athena Cut (Loose) vs True Background,$0.0499 \pm 0.0067$
18,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,trig_L2_calo_weta2,Model V10 (100 Rings) [Best SP] vs True Backgr...,$0.360 \pm 0.012$
19,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,trig_L2_calo_weta2,Model V10 (50 Rings) [Best SP] vs True Background,$0.355 \pm 0.011$
20,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,trig_L2_calo_weta2,Athena Cut (Loose) vs True Background,$0.2510 \pm 0.0068$
33,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,Eratio,Model V10 (100 Rings) [Best SP] vs True Backgr...,$0.453 \pm 0.011$
34,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,Eratio,Model V10 (50 Rings) [Best SP] vs True Background,$0.456 \pm 0.010$
35,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,Eratio,Athena Cut (Loose) vs True Background,$0.0843 \pm 0.0068$
36,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,trig_L2_calo_et,Model V10 (100 Rings) [Best SP] vs True Signal,$0.0589 \pm 0.0085$


Saved CSV:   /eos/home-i04/p/pmourafr/egamma_master/Plots/ConsolidatedAnalysis/Tables/consolidated_variable_js_distance.csv
Saved LaTeX: /eos/home-i04/p/pmourafr/egamma_master/Plots/ConsolidatedAnalysis/LaTeX/consolidated_variable_js_distance_latex_table.tex
\begin{table}[htbp]
\centering
\caption{Variable distribution JS distance ($\sqrt{D_{\text{JS}}}$) across comparisons.}
\label{tab:variable_js_distance}
\begin{tabular}{l c c l l l}
\toprule
ModelsCompared & ET & ETA & Feature & Comparison & JS Distance ($\sqrt{D_{\text{JS}}}$) \\
\midrule
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & 1 & 1 & trig\_L2\_calo\_et & Model V10 (100 Rings) [Best SP] vs True Background & $0.082 \pm 0.012$ \\
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & 1 & 1 & trig\_L2\_calo\_et & Model V10 (50 Rings) [Best SP] vs True Background & $0.078 \pm 0.010$ \\
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & 1 & 1 & trig\_L2\_calo\_et & Athena Cut (Loose) vs True Background & $0.0499 \pm 0.0067$ 

## 3. Group 2 — Quadrant Strategy Metrics (`*_quadrant_strategy_metrics.csv`)

In [21]:
# 3a. Load
df_strat = load_table_group("*_quadrant_strategy_metrics.csv")
display(df_strat.head(10))

[*_quadrant_strategy_metrics.csv] Discovered 2 files across 1 folder(s)


,ModelsCompared,ComparisonRun,ET,ETA,Strategy,PD,PD_unc,PF,PF_unc,SP,SP_unc,SourceFile
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Model V10 (100 Rings) [Best SP],0.802454,0.006300,0.135978,0.003364,0.832953,0.003609,et_1_eta_1_best_sp_vs_best_sp_quadrant_strateg...
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Model V10 (50 Rings) [Best SP],0.813721,0.006161,0.152061,0.003524,0.830742,0.003568,et_1_eta_1_best_sp_vs_best_sp_quadrant_strateg...
2,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,"Model V10 (100 Rings) [T(mu), Athena Loose PD=...",0.783425,0.006518,0.117874,0.003164,0.832044,0.003693,et_1_eta_1_pileup_vs_pileup_quadrant_strategy_...
3,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,"Model V10 (50 Rings) [T(mu), Athena Loose PD=7...",0.787431,0.006474,0.127889,0.003277,0.829230,0.003686,et_1_eta_1_pileup_vs_pileup_quadrant_strategy_...


In [22]:
# 3b. Filter
STRAT_MODELS_FILTER: Optional[list[str]] = None
STRAT_RUN_CONTAINS: Optional[str] = None

df_strat_filt = apply_common_filters(df_strat, models_filter=STRAT_MODELS_FILTER, run_contains=STRAT_RUN_CONTAINS)
display(df_strat_filt)

,ModelsCompared,ComparisonRun,ET,ETA,Strategy,PD,PD_unc,PF,PF_unc,SP,SP_unc,SourceFile
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Model V10 (100 Rings) [Best SP],0.802454,0.006300,0.135978,0.003364,0.832953,0.003609,et_1_eta_1_best_sp_vs_best_sp_quadrant_strateg...
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Model V10 (50 Rings) [Best SP],0.813721,0.006161,0.152061,0.003524,0.830742,0.003568,et_1_eta_1_best_sp_vs_best_sp_quadrant_strateg...
2,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,"Model V10 (100 Rings) [T(mu), Athena Loose PD=...",0.783425,0.006518,0.117874,0.003164,0.832044,0.003693,et_1_eta_1_pileup_vs_pileup_quadrant_strategy_...
3,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,"Model V10 (50 Rings) [T(mu), Athena Loose PD=7...",0.787431,0.006474,0.127889,0.003277,0.829230,0.003686,et_1_eta_1_pileup_vs_pileup_quadrant_strategy_...


In [23]:
# 3c. Format & Export LaTeX
if not df_strat_filt.empty:
    df_strat_pct = df_strat_filt.copy()
    for col in ["PD", "PD_unc", "PF", "PF_unc", "SP", "SP_unc"]:
        if col in df_strat_pct.columns:
            df_strat_pct[col] = df_strat_pct[col] * 100.0
    strat_keep = ["ModelsCompared", "ComparisonRun", "Strategy", "PD", "PD_unc", "PF", "PF_unc", "SP", "SP_unc"]
    strat_sub = df_strat_pct[[c for c in strat_keep if c in df_strat_pct.columns]]
    strat_fmt = LatexTableFormatter.merge_uncertainty_pairs(
        strat_sub,
        pairs={"PD [\%]": ("PD", "PD_unc"), "PF [\%]": ("PF", "PF_unc"), "SP [\%]": ("SP", "SP_unc")},
        precision=2,
        drop_original=True,
    )
    display(strat_fmt)
    strat_tex = save_group_outputs(
        df_strat_filt, strat_fmt, "quadrant_strategy_metrics",
        caption="Quadrant strategy performance metrics ($P_D$, $P_F$, $SP$).",
        label="tab:quadrant_strategy_metrics",
    )
    print(strat_tex)

<>:11: SyntaxWarning: invalid escape sequence '\%'
<>:11: SyntaxWarning: invalid escape sequence '\%'
<>:11: SyntaxWarning: invalid escape sequence '\%'
<>:11: SyntaxWarning: invalid escape sequence '\%'
<>:11: SyntaxWarning: invalid escape sequence '\%'
<>:11: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_10596/1639569769.py:11: SyntaxWarning: invalid escape sequence '\%'
  pairs={"PD [\%]": ("PD", "PD_unc"), "PF [\%]": ("PF", "PF_unc"), "SP [\%]": ("SP", "SP_unc")},
/tmp/ipykernel_10596/1639569769.py:11: SyntaxWarning: invalid escape sequence '\%'
  pairs={"PD [\%]": ("PD", "PD_unc"), "PF [\%]": ("PF", "PF_unc"), "SP [\%]": ("SP", "SP_unc")},
/tmp/ipykernel_10596/1639569769.py:11: SyntaxWarning: invalid escape sequence '\%'
  pairs={"PD [\%]": ("PD", "PD_unc"), "PF [\%]": ("PF", "PF_unc"), "SP [\%]": ("SP", "SP_unc")},


,ModelsCompared,ComparisonRun,Strategy,PD [\%],PF [\%],SP [\%]
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,Model V10 (100 Rings) [Best SP],$80.25 \pm 0.63$,$13.60 \pm 0.34$,$83.30 \pm 0.36$
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,Model V10 (50 Rings) [Best SP],$81.37 \pm 0.62$,$15.21 \pm 0.35$,$83.07 \pm 0.36$
2,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,"Model V10 (100 Rings) [T(mu), Athena Loose PD=...",$78.34 \pm 0.65$,$11.79 \pm 0.32$,$83.20 \pm 0.37$
3,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,"Model V10 (50 Rings) [T(mu), Athena Loose PD=7...",$78.74 \pm 0.65$,$12.79 \pm 0.33$,$82.92 \pm 0.37$


Saved CSV:   /eos/home-i04/p/pmourafr/egamma_master/Plots/ConsolidatedAnalysis/Tables/consolidated_quadrant_strategy_metrics.csv
Saved LaTeX: /eos/home-i04/p/pmourafr/egamma_master/Plots/ConsolidatedAnalysis/LaTeX/consolidated_quadrant_strategy_metrics_latex_table.tex
\begin{table}[htbp]
\centering
\caption{Quadrant strategy performance metrics ($P_D$, $P_F$, $SP$).}
\label{tab:quadrant_strategy_metrics}
\begin{tabular}{l l l l l l}
\toprule
ModelsCompared & ComparisonRun & Strategy & PD [\%] & PF [\%] & SP [\%] \\
\midrule
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & Comparison\_best\_sp\_vs\_best\_sp\_cross\_et\_1\_eta\_1 & Model V10 (100 Rings) [Best SP] & $80.25 \pm 0.63$ & $13.60 \pm 0.34$ & $83.30 \pm 0.36$ \\
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & Comparison\_best\_sp\_vs\_best\_sp\_cross\_et\_1\_eta\_1 & Model V10 (50 Rings) [Best SP] & $81.37 \pm 0.62$ & $15.21 \pm 0.35$ & $83.07 \pm 0.36$ \\
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & Comparison\_p

## 4. Group 3 — Quadrant Counts (`*_quadrant_counts.csv`)

In [24]:
# 4a. Load
df_counts = load_table_group("*_quadrant_counts.csv")
# display(df_counts.head(10))

[*_quadrant_counts.csv] Discovered 2 files across 1 folder(s)


In [25]:
# 4b. Filter
COUNTS_MODELS_FILTER: Optional[list[str]] = None
COUNTS_RUN_CONTAINS: Optional[str] = None

df_counts_filt = apply_common_filters(df_counts, models_filter=COUNTS_MODELS_FILTER, run_contains=COUNTS_RUN_CONTAINS)
display(df_counts_filt)

,ModelsCompared,ComparisonRun,ET,ETA,Quadrant,Count,Ratio,SourceFile
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Both Correct,11853,82.44%,et_1_eta_1_best_sp_vs_best_sp_quadrant_counts.csv
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Only Model V10 (100 Rings) [Best SP] Correct,324,2.25%,et_1_eta_1_best_sp_vs_best_sp_quadrant_counts.csv
2,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Only Model V10 (50 Rings) [Best SP] Correct,202,1.40%,et_1_eta_1_best_sp_vs_best_sp_quadrant_counts.csv
3,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,1,1,Both Wrong,1999,13.90%,et_1_eta_1_best_sp_vs_best_sp_quadrant_counts.csv
4,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,Both Correct,12006,83.50%,et_1_eta_1_pileup_vs_pileup_quadrant_counts.csv
5,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,"Only Model V10 (100 Rings) [T(mu), Athena Loos...",283,1.97%,et_1_eta_1_pileup_vs_pileup_quadrant_counts.csv
6,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,"Only Model V10 (50 Rings) [T(mu), Athena Loose...",195,1.36%,et_1_eta_1_pileup_vs_pileup_quadrant_counts.csv
7,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,Both Wrong,1894,13.17%,et_1_eta_1_pileup_vs_pileup_quadrant_counts.csv


In [26]:
# 4c. Format & Export LaTeX
if not df_counts_filt.empty:
    counts_keep = ["ModelsCompared", "ComparisonRun", "Quadrant", "Count", "Ratio"]
    counts_fmt = df_counts_filt[[c for c in counts_keep if c in df_counts_filt.columns]]
    display(counts_fmt)
    counts_tex = save_group_outputs(
        df_counts_filt, counts_fmt, "quadrant_counts",
        caption="Quadrant agreement counts and ratios across comparisons.",
        label="tab:quadrant_counts",
    )
    print(counts_tex)

,ModelsCompared,ComparisonRun,Quadrant,Count,Ratio
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,Both Correct,11853,82.44%
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,Only Model V10 (100 Rings) [Best SP] Correct,324,2.25%
2,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,Only Model V10 (50 Rings) [Best SP] Correct,202,1.40%
3,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_best_sp_vs_best_sp_cross_et_1_eta_1,Both Wrong,1999,13.90%
4,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,Both Correct,12006,83.50%
5,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,"Only Model V10 (100 Rings) [T(mu), Athena Loos...",283,1.97%
6,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,"Only Model V10 (50 Rings) [T(mu), Athena Loose...",195,1.36%
7,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,Both Wrong,1894,13.17%


Saved CSV:   /eos/home-i04/p/pmourafr/egamma_master/Plots/ConsolidatedAnalysis/Tables/consolidated_quadrant_counts.csv
Saved LaTeX: /eos/home-i04/p/pmourafr/egamma_master/Plots/ConsolidatedAnalysis/LaTeX/consolidated_quadrant_counts_latex_table.tex
\begin{table}[htbp]
\centering
\caption{Quadrant agreement counts and ratios across comparisons.}
\label{tab:quadrant_counts}
\begin{tabular}{l l l c l}
\toprule
ModelsCompared & ComparisonRun & Quadrant & Count & Ratio \\
\midrule
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & Comparison\_best\_sp\_vs\_best\_sp\_cross\_et\_1\_eta\_1 & Both Correct & 11853 & 82.44% \\
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & Comparison\_best\_sp\_vs\_best\_sp\_cross\_et\_1\_eta\_1 & Only Model V10 (100 Rings) [Best SP] Correct & 324 & 2.25% \\
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & Comparison\_best\_sp\_vs\_best\_sp\_cross\_et\_1\_eta\_1 & Only Model V10 (50 Rings) [Best SP] Correct & 202 & 1.40% \\
Model\_V10\_100\_Rings\_vs\_Mo

## 5. Group 4 — Pileup Fit & Efficiency vs $\langle\mu\rangle$ (`*_pileup_*.csv`)

In [27]:
# 5a. Load
df_pileup_fit = load_table_group("*_pileup_fit_parameters.csv")
df_pileup_eff = load_table_group("*_pileup_efficiency_vs_mu_*.csv")
display(df_pileup_fit)
display(df_pileup_eff.head(10))

[*_pileup_fit_parameters.csv] Discovered 1 files across 1 folder(s)


[*_pileup_efficiency_vs_mu_*.csv] Discovered 2 files across 1 folder(s)


,ModelsCompared,ComparisonRun,ET,ETA,Strategy,Target PD (%),Slope (a),Intercept (b),R^2,Fit Formula,SourceFile
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,"Model V10 (100 Rings) [T(mu), Athena Loose PD=...",77.90%,-0.0041,-0.6015,0.4389,T(mu) = -0.0041*mu - 0.6015,et_1_eta_1_pileup_vs_pileup_pileup_fit_paramet...
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,"Model V10 (50 Rings) [T(mu), Athena Loose PD=7...",77.90%,-0.0010,-0.5030,0.1106,T(mu) = -0.0010*mu - 0.5030,et_1_eta_1_pileup_vs_pileup_pileup_fit_paramet...


,ModelsCompared,ComparisonRun,ET,ETA,mu_bin_center,"Model V10 (100 Rings) [T(mu), Athena Loose PD=77.90%]_eff","Model V10 (100 Rings) [T(mu), Athena Loose PD=77.90%]_unc","Model V10 (50 Rings) [T(mu), Athena Loose PD=77.90%]_eff","Model V10 (50 Rings) [T(mu), Athena Loose PD=77.90%]_unc",Athena Cut (Loose)_eff,Athena Cut (Loose)_unc,SourceFile
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,13.55,0.047619,0.026830,0.079365,0.034056,0.380952,0.061182,et_1_eta_1_pileup_vs_pileup_pileup_efficiency_...
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,21.65,0.102000,0.013535,0.128000,0.014941,0.434000,0.022165,et_1_eta_1_pileup_vs_pileup_pileup_efficiency_...
2,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,29.75,0.100932,0.011870,0.121118,0.012857,0.416149,0.019424,et_1_eta_1_pileup_vs_pileup_pileup_efficiency_...
3,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,37.85,0.119874,0.012900,0.132492,0.013464,0.438486,0.019707,et_1_eta_1_pileup_vs_pileup_pileup_efficiency_...
4,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,45.95,0.110485,0.010526,0.124014,0.011067,0.388952,0.016369,et_1_eta_1_pileup_vs_pileup_pileup_efficiency_...
5,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,54.05,0.114329,0.007173,0.128049,0.007532,0.411077,0.011091,et_1_eta_1_pileup_vs_pileup_pileup_efficiency_...
6,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,62.15,0.120773,0.005338,0.126141,0.005439,0.421095,0.008089,et_1_eta_1_pileup_vs_pileup_pileup_efficiency_...
7,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,70.25,0.125085,0.008649,0.133288,0.008886,0.405332,0.012836,et_1_eta_1_pileup_vs_pileup_pileup_efficiency_...
8,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,78.35,0.124224,0.018381,0.118012,0.017979,0.403727,0.027343,et_1_eta_1_pileup_vs_pileup_pileup_efficiency_...
9,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,86.45,0.186441,0.029274,0.180791,0.028927,0.429379,0.037206,et_1_eta_1_pileup_vs_pileup_pileup_efficiency_...


In [28]:
# 5b. Filter
PILEUP_MODELS_FILTER: Optional[list[str]] = None
PILEUP_RUN_CONTAINS: Optional[str] = None

df_pileup_fit_filt = apply_common_filters(df_pileup_fit, models_filter=PILEUP_MODELS_FILTER, run_contains=PILEUP_RUN_CONTAINS)
df_pileup_eff_filt = apply_common_filters(df_pileup_eff, models_filter=PILEUP_MODELS_FILTER, run_contains=PILEUP_RUN_CONTAINS)
display(df_pileup_fit_filt)

,ModelsCompared,ComparisonRun,ET,ETA,Strategy,Target PD (%),Slope (a),Intercept (b),R^2,Fit Formula,SourceFile
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,"Model V10 (100 Rings) [T(mu), Athena Loose PD=...",77.90%,-0.0041,-0.6015,0.4389,T(mu) = -0.0041*mu - 0.6015,et_1_eta_1_pileup_vs_pileup_pileup_fit_paramet...
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,Comparison_pileup_vs_pileup_cross_et_1_eta_1,1,1,"Model V10 (50 Rings) [T(mu), Athena Loose PD=7...",77.90%,-0.0010,-0.5030,0.1106,T(mu) = -0.0010*mu - 0.5030,et_1_eta_1_pileup_vs_pileup_pileup_fit_paramet...


In [29]:
# 5c. Format & Export LaTeX
if not df_pileup_fit_filt.empty:
    fit_keep = ["ModelsCompared", "ET", "ETA", "Strategy", "Target PD (%)", "Slope (a)", "Intercept (b)", "R^2", "Fit Formula"]
    fit_fmt = df_pileup_fit_filt[[c for c in fit_keep if c in df_pileup_fit_filt.columns]]
    display(fit_fmt)
    fit_tex = save_group_outputs(
        df_pileup_fit_filt, fit_fmt, "pileup_fit_parameters",
        caption="Linear pileup threshold calibration parameters $T(\\mu) = a\\mu + b$.",
        label="tab:pileup_fit_parameters",
    )
    print(fit_tex)

,ModelsCompared,ET,ETA,Strategy,Target PD (%),Slope (a),Intercept (b),R^2,Fit Formula
0,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,"Model V10 (100 Rings) [T(mu), Athena Loose PD=...",77.90%,-0.0041,-0.6015,0.4389,T(mu) = -0.0041*mu - 0.6015
1,Model_V10_100_Rings_vs_Model_V10_50_Rings,1,1,"Model V10 (50 Rings) [T(mu), Athena Loose PD=7...",77.90%,-0.0010,-0.5030,0.1106,T(mu) = -0.0010*mu - 0.5030


Saved CSV:   /eos/home-i04/p/pmourafr/egamma_master/Plots/ConsolidatedAnalysis/Tables/consolidated_pileup_fit_parameters.csv
Saved LaTeX: /eos/home-i04/p/pmourafr/egamma_master/Plots/ConsolidatedAnalysis/LaTeX/consolidated_pileup_fit_parameters_latex_table.tex
\begin{table}[htbp]
\centering
\caption{Linear pileup threshold calibration parameters $T(\mu) = a\mu + b$.}
\label{tab:pileup_fit_parameters}
\begin{tabular}{l c c l l c c c l}
\toprule
ModelsCompared & ET & ETA & Strategy & Target PD (%) & Slope (a) & Intercept (b) & R^2 & Fit Formula \\
\midrule
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & 1 & 1 & Model V10 (100 Rings) [T(mu), Athena Loose PD=77.90%] & 77.90% & -0.004100 & -0.601500 & 0.438900 & T(mu) = -0.0041*mu - 0.6015 \\
Model\_V10\_100\_Rings\_vs\_Model\_V10\_50\_Rings & 1 & 1 & Model V10 (50 Rings) [T(mu), Athena Loose PD=77.90%] & 77.90% & -0.001000 & -0.503000 & 0.110600 & T(mu) = -0.0010*mu - 0.5030 \\
\bottomrule
\end{tabular}
\end{table}

